### API Key

To obtain your own API key, go to the [NTA Developer Portal](https://developer.nationaltransport.ie/) and create an account by clicking the **Sign Up** button.

You can register using your `@factored.ai` email address. Once logged in, navigate to the **Products** tab and select **GTFS Realtime**.  
Choose a name for the product and subscribe to it. After subscribing, you’ll find your two API keys under the **Profile** tab.

---

### Storing your API Key (Best Practice)

It is considered best practice not to hard-code your API key directly in your codebase.  
Instead, store it in your `local_settings.py` file and import it when needed:


```python
# local_settings.py
API_KEY = "<your-api-key>"
```



### API Routes

This server exposes the following endpoint:

| Endpoint                | Description                  |
|------------------------|------------------------------|
| `/api/v1/arrivals`     | Returns arrival information    |

Before you can query these routes, you’ll need to start the server locally.  
Run the following commands in your terminal:

```bash
# Create and activate a virtual environment
python3 -m venv my_venv
source my_venv/bin/activate

# Install dependencies
pip install -r requirements.txt

# Run the server
python3 server.py


In [43]:
import settings
import local_settings
import gtfs

In [44]:
import requests
import pandas as pd
import json
import os
import zipfile
import io
from pathlib import Path
from datetime import datetime


In [ ]:
def get_files_from_path(directory: str, extension: str = '.txt') -> list:
    '''
    Returns a list of all files in a directory with a given extension.
    '''
    filenames = []
    for entry in os.listdir(directory):
        full_path = os.path.join(directory, entry)
        _, _extension = os.path.splitext(full_path)
        if os.path.isfile(full_path) and _extension == extension:
            filenames.append(full_path)
    return filenames

def generate_df_from_txt(filename: str):
    '''
    Reads a txt file and [always] returns a pandas DataFrame even if the file is empty.
    If the file is empty, the DataFrame will have 0 rows.
    '''
    try:
        df = pd.read_csv(filename)
        return df
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return pd.DataFrame()

In [ ]:
# This is the base URL of the server that will be used to make requests to the API
BASE_URL = "http://localhost:7341"

# Assuming your API key is stored in a local_settings.py file in the same directory
api_key = local_settings.API_KEY
header = {"X-API-KEY": api_key, "Accept": "application/json"}

try:
    # TODO:
    # 1. Load the available stop numbers from static GTFS data (e.g., stops.txt or via a GTFS utility function).
    #   1.1 Put the static data in a folder called `static`
    # 2. Decide which stops to query—either all, a sample, or by filtering on criteria (e.g., first N stops, or a specific route).
    # 3. Construct a stop_list (list of stop numbers as strings).
    # 4. Use the stop_list to call the /api/v1/arrivals endpoint.
    # 
    # r = requests.get(f"{BASE_URL}/api/v1/arrivals", params={"stop": stop_list}, headers=header)
    stop_id = "1508"
    r = requests.get(f"{BASE_URL}/api/v1/arrivals?stop={stop_id}", headers=header)
    r.raise_for_status()
    data = r.json()
    print("Data was retrieved successfully")
except requests.exceptions.HTTPError as e:
    print(f'HTTP error occurred: {e}')
except requests.exceptions.ConnectionError as e:
    print(f'Connection error occurred: {e}')
except requests.exceptions.RequestException as e:
    print(f'An unexpected error occurred: {e}')

Data was retrieved successfully


In [9]:
print(f"Request data for stop {stop_id}")
data

Request data for stop 1508


{'1508': {'arrivals': [{'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T10:53:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arrival': None,
    'route': 'F1',
    'scheduled_arrival': '2025-10-28T11:03:09'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F2',
    'scheduled_arrival': '2025-10-28T11:04:47'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T11:08:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Tyrrelstown',
    'real_time_arrival': None,
    'route': '40D',
    'scheduled_arrival': '2025-10-28T11:14:12'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arr

## Mobility Database API Access

This section demonstrates how to authenticate and access the [Mobility Database API](https://mobilitydatabase.org/) to fetch GTFS feed information programmatically.

---

### Step 1: Create an Account

Before you can use the API, you need to create an account:

1. Visit the [Mobility Database website](https://mobilitydatabase.org/)
2. Click on **Sign Up** or **Create Account**
3. You can register using your **`@factored.ai`** email address
4. Complete the registration process and verify your email if needed

---

### Step 2: Obtain Your Refresh Token

Once your account is created and verified:

1. Log in to your Mobility Database account
2. Navigate to your **Account** page
3. Generate or copy your **Refresh Token**
4. Store this token securely (do not commit it to version control)

**Best Practice:** Store your refresh token in a `local_settings.py` file:

```python
# local_settings.py
MOBILITY_DB_REFRESH_TOKEN = "your_refresh_token_here"
```

---

### Step 3: Generate an Access Token (POST Request)

The refresh token is used to generate a short-lived **access token** that you'll use for API requests:

- **Method:** `POST`
- **Endpoint:** `https://api.mobilitydatabase.org/v1/tokens`
- **Headers:** `Content-Type: application/json`
- **Body:** `{"refresh_token": "your_refresh_token"}`

This returns an `access_token` that is valid for a limited time (1 hour).

---

### Step 4: Make API Requests (GET Request)

Use the access token to make authenticated GET requests to fetch data:

- **Method:** `GET`
- **Headers:** `Authorization: Bearer {access_token}`
- **Example Endpoint:** `https://api.mobilitydatabase.org/v1/gtfs_feeds/{feed_id}`

---

### Available Endpoints

- **List all feeds:** `GET /v1/gtfs_feeds`
- **Get specific feed:** `GET /v1/gtfs_feeds/{feed_id}`
- **Search feeds:** `GET /v1/gtfs_feeds?filter={criteria}`

For more details, refer to the [Mobility Database API Documentation](https://mobilitydatabase.org/api-docs).




In [ ]:
# Step 1: Generate access token using POST (required for authentication)
headers = {'Content-Type': 'application/json'}
data = {"refresh_token": local_settings.REFRESH_TOKEN}
r = requests.post("https://api.mobilitydatabase.org/v1/tokens", headers=headers, json=data)

print(f"Status Code: {r.status_code}")
if r.status_code == 200:
    token_response = r.json()
    access_token = token_response.get('access_token')
    print(f"Access Token obtained successfully!")
    print(f"Access Token: {access_token[:50]}..." if access_token else "N/A")
else:
    print(f"Error: {r.text}")
    access_token = None


Status Code: 200
✓ Access Token obtained successfully!
Access Token: eyJhbGciOiJSUzI1NiIsImtpZCI6IjdlYTA5ZDA1NzI2MmU2M2...


---

### Example: Fetching Transport for Ireland Data

The Transport for Ireland GTFS feed has the ID **`mdb-2364`**. You can fetch its details using the authenticated API calls demonstrated below.

In [ ]:
# Step 2: Use the access token to make GET requests
# Example: Fetching GTFS feeds data

if access_token:
    # Use Authorization header with Bearer token for GET requests
    auth_headers = {
        'Authorization': f'Bearer {access_token}',
        'Accept': 'application/json'
    }
    
    # Example GET request to fetch feeds
    feed_id = "mdb-2364"  # Transport for Ireland GTFS feed
    response = requests.get(
        f"https://api.mobilitydatabase.org/v1/gtfs_feeds/{feed_id}",
        headers=auth_headers
    )
    
    print(f"GET Request Status Code: {response.status_code}")
    if response.status_code == 200:
        feed_data = response.json()
        print(f"Data fetched successfully!")
        print(json.dumps(feed_data, indent=2))
    else:
        print(f"Error: {response.text}")
else:
    print("No access token available. Run the previous cell first.")


GET Request Status Code: 200
✓ Data fetched successfully!
{
  "id": "mdb-2364",
  "data_type": "gtfs",
  "created_at": "2025-02-10T20:16:06.599956Z",
  "external_ids": [
    {
      "external_id": "2364",
      "source": "mdb"
    }
  ],
  "provider": "Transport for Ireland (TFI)",
  "feed_contact_email": "",
  "source_info": {
    "producer_url": "https://www.transportforireland.ie/transitData/Data/GTFS_All.zip",
    "authentication_type": 0,
    "authentication_info_url": "",
    "api_key_parameter_name": "",
    "license_url": ""
  },
  "redirects": [],
  "status": "active",
  "official": null,
  "official_updated_at": null,
  "feed_name": "Aggregate Ireland feed from National Transport Authority",
  "note": "",
  "locations": [
    {
      "country_code": "GB",
      "country": "United Kingdom",
      "subdivision_name": "Northern Ireland",
      "municipality": "Northern Ireland"
    },
    {
      "country_code": "IE",
      "country": "Ireland",
      "subdivision_name": "Leinst

In [ ]:
# Extract the download URL from the feed data
if access_token and 'feed_data' in locals() and feed_data:
    # Look for the latest GTFS file URL
    latest_url = feed_data.get('latest_dataset', {}).get('hosted_url') or \
        feed_data.get('source_info', {}).get('producer_url')
    
    print(f"Feed Name: {feed_data.get('data_type')} - {feed_data.get('provider', 'N/A')}")
    print(f"Feed ID: {feed_data.get('id', 'N/A')}")
    print(f"Latest GTFS URL: {latest_url}")
    
    # Display other useful metadata
    if 'latest_dataset' in feed_data:
        latest = feed_data['latest_dataset']
        print(f"\nDataset Info:")
        print(f"  - Downloaded at: {latest.get('downloaded_at', 'N/A')}")
        print(f"  - Bounding box: {latest.get('bounding_box', 'N/A')}")
else:
    print("No feed data available. Run the previous cells first.")
    latest_url = None


Feed Name: gtfs - Transport for Ireland (TFI)
Feed ID: mdb-2364
Latest GTFS URL: https://files.mobilitydatabase.org/mdb-2364/mdb-2364-202510290020/mdb-2364-202510290020.zip

Dataset Info:
  - Downloaded at: 2025-10-29T00:23:13.824275Z
  - Bounding box: {'minimum_latitude': 51.45159, 'maximum_latitude': 55.37705, 'minimum_longitude': -10.46157, 'maximum_longitude': -5.93626793243424}


In [ ]:
# Download and extract the GTFS zip file
if latest_url:
    now = datetime.now()
    print(f"Downloading GTFS data from: {latest_url} at {now}")
    
    # Create a directory for extracted files
    extract_dir = Path("data")
    extract_dir.mkdir(exist_ok=True)

    timestamp_file = f'{extract_dir}/timestamp.txt'

    # Download the zip file in chunks
    print(f"Download started at {now}")
    response = requests.get(latest_url, stream=True)
    
    if response.status_code == 200:
        print("Download successful!")
        # Extract the zip file
        # response.content is the raw bytes of the zip file
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            zip_ref.extractall(extract_dir)
            print(f"Extracted to: {extract_dir}/")
            print(f"\nExtracted files:")

            # Finds all the txt files in the extracted directory
            for file in sorted(extract_dir.glob("*.txt")):
                file_size = file.stat().st_size / 1024  # Size in KB
                print(f"  - {file.name} ({file_size:.1f} KB)")

        # Save the timestamp of the most recent download
        with open(timestamp_file, 'w') as f:
            f.write(f"{now}")

    else:
        print(f"Download failed with status code: {response.status_code}")
        extract_dir = None
else:
    print("No URL available to download.")
    extract_dir = None


Download started at 2025-10-29 16:18:22.909576
Download successful!
Extracted to: data/

Extracted files:
  - agency.txt (7.4 KB)
  - cache_info.txt (0.0 KB)
  - calendar.txt (11.5 KB)
  - calendar_dates.txt (34.7 KB)
  - feed_info.txt (0.3 KB)
  - routes.txt (51.8 KB)
  - shapes.txt (313714.0 KB)
  - stop_times.txt (350482.3 KB)
  - stops.txt (836.5 KB)
  - timestamp.txt (0.0 KB)
  - trips.txt (16243.7 KB)


In [ ]:
dataframes = {}
print(f"Generating DataFrames from GTFS data...")

for file in get_files_from_path(extract_dir):
    try:
        df_name = file.replace(f'{extract_dir}/', '').replace('.txt', '')
        df = generate_df_from_txt(file)
        dataframes[df_name] = df
        print('\n')
        print(f"{df_name}: {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
    except Exception as e:
        print(f"Error generating DataFrame from {file}: {e}")

# Display the first few rows of each dataframe
print("\n" + "="*50)
print("SAMPLE DATA FROM EACH DATAFRAME:")
print("="*50)

for name, df in dataframes.items():
    print(f"\n{name.upper()}:")
    print("-" * 30)
    print(df.head(3))
    print(f"Shape: {df.shape}")
    print()

Generating DataFrames from GTFS data...


agency: 101 rows, 4 columns
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone']


calendar_dates: 2284 rows, 3 columns
Columns: ['service_id', 'date', 'exception_type']


stop_times: 6842290 rows, 9 columns
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'timepoint']


cache_info: 2 rows, 1 columns
Columns: ['{']


shapes: 6889576 rows, 5 columns
Columns: ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled']


timestamp: 0 rows, 1 columns
Columns: ['2025-10-29 16:18:22.909576']


trips: 200209 rows, 8 columns
Columns: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id']


feed_info: 1 rows, 7 columns
Columns: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version', 'feed_contact_email']


stops: 

### Exploring the GTFS Data

Now that we have the data loaded, let's explore the key datasets:

## TODO

- Make updates to gtfs.py so it doesn't pull static data from the NTA API directly but from the Mobility DB
- Update requirements.txt